# Giai đoạn 1: Cluster-CSKGO trên Feature Embedding (Global Imbalance + Non-IID)

Notebook thực nghiệm Giai đoạn 1 kết hợp **Client Clustering trên Centroid Embedding** và thuật toán **CSKGO** (K-Means kết hợp độ chặt/tách biệt + Gaussian Oversampling) trên không gian ResNet-18 embedding (`512` chiều).

### Luồng hoạt động
1. **Phân cụm 1 lần (`centroid_elbow`):** Mỗi client tính và gửi vector centroid embedding chuẩn hóa $L_2$. Hệ thống chạy **Spherical K-Means + Elbow** để tự xác định số cụm $K_{\text{cluster}}$ và gom các client thành các *super-client* (clusters).
2. **Kiểm tra ngưỡng $T$ & CSKGO tại Cluster:** Mỗi cluster đối chiếu số mẫu hiện có của từng class với ngưỡng cố định `CLUSTER_CLASS_THRESHOLD` ($T$) do Server quy định:
   - $N_{m,c} = 0$: bỏ qua.
   - $1 \le N_{m,c} < 20$ (`MIN_SAMPLES_FOR_CSKMEANS`): xem là 1 cụm ($k=1$) và chạy **GOS** trực tiếp.
   - $20 \le N_{m,c} < T$: chạy đầy đủ **CSK-means + SRA + GOS** để sinh bù mẫu thiếu.
   - $N_{m,c} \ge T$: giữ nguyên.
3. **Huấn luyện & Đánh giá:** Hỗ trợ đối chứng giữa `EXPERIMENT_MODE = 'cluster_cskgo'` vs `'global_cskgo'` (bỏ bước cluster), và hai kiểu huấn luyện `TRAIN_PARADIGM = 'centralized'` vs `'cluster_fedavg'`.


## Bảng các tham số tự set tay (Cell 2)

| Nhóm | Tham số | Ý nghĩa | Giá trị gợi ý (CIFAR-10) |
| :--- | :--- | :--- | :--- |
| **Thực nghiệm** | `EXPERIMENT_MODE` | `'cluster_cskgo'` (phân cụm rồi CSKGO tại cluster) hoặc `'global_cskgo'` (bỏ cluster, chạy CSKGO 1 lần trên toàn bộ data) | `'cluster_cskgo'` |
| | `TRAIN_PARADIGM` | `'centralized'` (gộp data về Server train) hoặc `'cluster_fedavg'` (mỗi cluster train local rồi FedAvg lên Server) | `'centralized'` |
| | `RUN_ID` | ID lượt chạy để lưu tách biệt thư mục kết quả (`cskgo_stage1/run{RUN_ID}`) | `1` |
| **Dữ liệu & Partition** | `DATASET_NAME` | Tên dataset: `'cifar10'` hoặc `'mnist'` | `'cifar10'` |
| | `IMBALANCE_MODE` | Chế độ mất cân bằng của partition: `'Global'` hoặc `'None'` (phải khớp với `partition_dirichlet.ipynb`) | `'Global'` |
| | `PARTITION_TYPE` | Kiểu chia dữ liệu: `'dirichlet'` hoặc `'noniid_N'` | `'dirichlet'` |
| | `DIRICHLET_ALPHA` | Độ lệch Non-IID Dirichlet (`0.1`, `0.5`, `1.0`) | `0.1` |
| | `NUM_CLIENTS` | Tổng số client thiết bị biên | `25` |
| | `RESNET_SIZE` | Kích thước ảnh của cached ResNet-18 embedding (`112` hoặc `224`) | `112` |
| **Phân cụm Client** | `K_MAX` | Số cụm tối đa khi quét Elbow trên các client centroid | `15` |
| **Ngưỡng & CSKGO** | `CLUSTER_CLASS_THRESHOLD` | Ngưỡng $T$: số mẫu tối thiểu mỗi class cần đạt tại 1 cluster | `300` |
| | `GLOBAL_TARGET_COUNT` | Mục tiêu số mẫu/class khi chạy `global_cskgo` (`None` = tự lấy bằng lớp đa số toàn cục) | `None` |
| | `MIN_SAMPLES_FOR_CSKMEANS` | Ngưỡng tối thiểu để chạy CSK-means ($< 20$ dùng GOS 1 cụm) | `20` |
| | `MAX_OVERSAMPLE_RATIO` | Trần tỷ lệ nhân mẫu tối đa $R_{\max}$ (`None` nếu muốn sinh thẳng lên đúng ngưỡng $T$) | `10` |
| | `CSKGO_BETA` | Hệ số bán kính lân cận $\beta$ trong chỉ số tách biệt `sep` (theo paper = `0.7`) | `0.7` |
| | `COV_REG` | Hệ số $\lambda$ cộng vào đường chéo $\Sigma + \lambda I$ để tránh suy biến ma trận trong GOS | `1e-4` |
| **Huấn luyện** | `EPOCHS` | Số epoch train trên Server (dùng khi `TRAIN_PARADIGM = 'centralized'`) | `40` |
| | `ROUNDS` | Số round FL (dùng khi `TRAIN_PARADIGM = 'cluster_fedavg'`) | `40` |
| | `LOCAL_EPOCHS` | Số epoch local tại mỗi cluster trong 1 round (dùng khi `TRAIN_PARADIGM = 'cluster_fedavg'`) | `3` |
| | `BATCH_SIZE` | Kích thước mini-batch khi huấn luyện `LinearProbe` | `64` |
| | `LEARNING_RATE` | Learning rate của SGD | `0.01` |
| | `EVAL_BATCH_SIZE` | Kích thước batch khi đánh giá | `4096` |
| **Seeds** | `PARTITION_SEED`, `MODEL_SEED`, `BATCH_SEED`, `CLUSTER_SEED`, `CSKGO_SEED` | Các seed cố định giúp tái lập kết quả giữa các lần chạy | `42`, `42`, `42`, `44`, `45` |


In [ ]:
import csv
import hashlib
import json
import os
import pickle
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Enable GPU in Colab for faster execution.')

# ── Cấu hình chính (chỉnh sửa các tham số tự set tại đây) ─────────────────────
EXPERIMENT_MODE = 'cluster_cskgo'   # 'cluster_cskgo' hoặc 'global_cskgo'
TRAIN_PARADIGM  = 'centralized'     # 'centralized' hoặc 'cluster_fedavg'
RUN_ID = 1

DATASET_NAME    = 'cifar10'         # 'cifar10' hoặc 'mnist'
IMBALANCE_MODE  = 'Global'          # 'Global' hoặc 'None'
PARTITION_TYPE  = 'dirichlet'       # 'dirichlet' hoặc 'noniid_N'
DIRICHLET_ALPHA = 0.1
NUM_CLIENTS     = 25
RESNET_SIZE     = 112               # 112 hoặc 224

K_MAX = 15

CLUSTER_CLASS_THRESHOLD  = 300      # Ngưỡng T tại mỗi cluster
GLOBAL_TARGET_COUNT      = None     # None => dùng max class count khi EXPERIMENT_MODE='global_cskgo'
MIN_SAMPLES_FOR_CSKMEANS = 20       # < 20 mẫu dùng GOS 1 cụm (k=1)
MAX_OVERSAMPLE_RATIO     = 10       # None nếu không giới hạn trần nhân bản
CSKGO_BETA               = 0.7
COV_REG                  = 1e-4

EPOCHS          = 40                # dùng cho TRAIN_PARADIGM == 'centralized'
ROUNDS          = 40                # dùng cho TRAIN_PARADIGM == 'cluster_fedavg'
LOCAL_EPOCHS    = 3                 # dùng cho TRAIN_PARADIGM == 'cluster_fedavg'
BATCH_SIZE      = 64
LEARNING_RATE   = 0.01
EVAL_BATCH_SIZE = 4096

PARTITION_SEED = 42
MODEL_SEED     = 42
BATCH_SEED     = 42
CLUSTER_SEED   = 44
CSKGO_SEED     = 45

# ── Validate cấu hình ──────────────────────────────────────────────────────────
if EXPERIMENT_MODE not in {'cluster_cskgo', 'global_cskgo'}:
    raise ValueError("EXPERIMENT_MODE phải là 'cluster_cskgo' hoặc 'global_cskgo'.")
if TRAIN_PARADIGM not in {'centralized', 'cluster_fedavg'}:
    raise ValueError("TRAIN_PARADIGM phải là 'centralized' hoặc 'cluster_fedavg'.")
if DATASET_NAME not in {'cifar10', 'mnist'}:
    raise ValueError("DATASET_NAME phải là 'cifar10' hoặc 'mnist'.")
if str(IMBALANCE_MODE).strip().lower() not in {'global', 'none'}:
    raise ValueError("IMBALANCE_MODE phải là 'Global' hoặc 'None'.")

# ── Đường dẫn Google Drive / Local fallback ────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FLasIRL(new)')
except ImportError:
    print('Not running in Colab; using local directory fallback.')
    PROJECT_ROOT = Path.cwd() / 'FLasIRL(new)' if not Path('/content').exists() else Path('/content/FLasIRL(new)')

if not PROJECT_ROOT.parent.exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / 'data' / DATASET_NAME
PARTITION_DIR = DATA_DIR / 'partitions' / 'Dirichlet'
EXP_DIR = PROJECT_ROOT / 'cskgo_stage1'

run_folder_name = str(RUN_ID) if str(RUN_ID).startswith('run') else f'run{RUN_ID}'
RUN_DIR = EXP_DIR / run_folder_name
for folder in [DATA_DIR, PARTITION_DIR, EXP_DIR, RUN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

METRICS_PATH     = RUN_DIR / 'metrics.csv'
CONFIG_PATH      = RUN_DIR / 'config.json'
CSKGO_STATS_PATH = RUN_DIR / 'cskgo_stats.json'
SUMMARY_PATH     = RUN_DIR / 'summary.json'
FINAL_MODEL_PATH = RUN_DIR / 'final_model_parameters.pt'

print('EXPERIMENT_MODE:', EXPERIMENT_MODE, '| TRAIN_PARADIGM:', TRAIN_PARADIGM)
print('RUN_DIR        :', RUN_DIR)


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _load_cifar10_labels(data_dir: Path):
    batch_dir = data_dir / 'cifar-10-batches-py'
    if not batch_dir.is_dir():
        batch_dir = data_dir / 'batches-py'
    if not batch_dir.is_dir():
        raise FileNotFoundError(f'CIFAR-10 batches-py không tồn tại: {data_dir}')

    def read_labels(path: Path):
        with path.open('rb') as handle:
            batch = pickle.load(handle, encoding='bytes')
        return torch.tensor(batch.get(b'labels', batch.get('labels')), dtype=torch.long)

    train_labels = torch.cat([read_labels(batch_dir / f'data_batch_{i}') for i in range(1, 6)])
    test_labels = read_labels(batch_dir / 'test_batch')
    return train_labels, test_labels


def _parse_partition_type(partition_type):
    if partition_type == 'dirichlet':
        return 'dirichlet', None
    match = re.fullmatch(r'noniid_(\d+)', partition_type)
    if match:
        return 'noniid', int(match.group(1))
    raise ValueError(f"PARTITION_TYPE không hợp lệ: '{partition_type}'.")


def embedding_cache_filename(dataset_name, resnet_size, split):
    key = f'{dataset_name}_resnet18_imagenet1k_v1_resize{resnet_size}_imagenet_norm_{split}'
    h = hashlib.sha256(key.encode()).hexdigest()[:8]
    return f'resnet18_sz{resnet_size}_{split}_{h}.pt'


if DATASET_NAME == 'mnist':
    train_ds = datasets.MNIST(DATA_DIR, train=True, download=False)
    test_ds = datasets.MNIST(DATA_DIR, train=False, download=False)
    train_labels = train_ds.targets.long()
    test_labels = test_ds.targets.long()
else:
    train_labels, test_labels = _load_cifar10_labels(DATA_DIR)

NUM_CLASSES = int(train_labels.max().item()) + 1

partition_kind, classes_per_client = _parse_partition_type(PARTITION_TYPE)
if partition_kind == 'dirichlet':
    stem = f'{DATASET_NAME}_dirichlet_alpha{DIRICHLET_ALPHA}_{NUM_CLIENTS}_seed{PARTITION_SEED}'
else:
    stem = f'noniid_{classes_per_client}_{NUM_CLIENTS}clients_seed{PARTITION_SEED}'
if str(IMBALANCE_MODE).strip().lower() == 'global':
    stem += '_global'
partition_path = PARTITION_DIR / f'{stem}.pt'

if not partition_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy partition: {partition_path}. '
        'Hãy chạy partition_dirichlet.ipynb trước.'
    )

partition = torch.load(partition_path, map_location='cpu', weights_only=False)
client_indices = partition['client_indices']
if len(client_indices) != NUM_CLIENTS:
    raise ValueError(f'Partition có {len(client_indices)} clients, nhưng NUM_CLIENTS={NUM_CLIENTS}.')

partition_num_samples = sum(len(idx) for idx in client_indices)
if partition_num_samples == 0 or len(torch.cat(client_indices).unique()) != partition_num_samples:
    raise ValueError('Partition chứa chỉ số mẫu trùng lặp hoặc rỗng.')

train_embed_path = DATA_DIR / embedding_cache_filename(DATASET_NAME, RESNET_SIZE, 'train')
test_embed_path = DATA_DIR / embedding_cache_filename(DATASET_NAME, RESNET_SIZE, 'test')
if not (train_embed_path.exists() and test_embed_path.exists()):
    raise FileNotFoundError(
        f'Thiếu embedding cache:\n  - {train_embed_path}\n  - {test_embed_path}\n'
        'Hãy bật GENERATE_EMBEDDINGS=True trong partition_dirichlet.ipynb để tạo trước.'
    )

train_features = torch.load(train_embed_path, map_location='cpu', weights_only=True).float().to(DEVICE)
test_features = torch.load(test_embed_path, map_location='cpu', weights_only=True).float().to(DEVICE)
train_labels_gpu = train_labels.to(DEVICE)
test_labels_gpu = test_labels.to(DEVICE)
EMBEDDING_DIM = train_features.shape[1]

# Xác định nhóm lớp đa số / thiểu số toàn cục từ dữ liệu trong partition để thống kê Recall
all_partition_indices = torch.cat(client_indices)
global_class_counts = torch.bincount(train_labels[all_partition_indices], minlength=NUM_CLASSES).numpy()
median_count = float(np.median(global_class_counts))
MINORITY_CLASSES = [c for c in range(NUM_CLASSES) if global_class_counts[c] < median_count]
MAJORITY_CLASSES = [c for c in range(NUM_CLASSES) if global_class_counts[c] >= median_count]

print(f'Loaded embeddings: train={tuple(train_features.shape)}, test={tuple(test_features.shape)}')
print('Global class counts in partition:', {c: int(global_class_counts[c]) for c in range(NUM_CLASSES)})
print('Minority classes (< median):', MINORITY_CLASSES, '| Majority classes:', MAJORITY_CLASSES)


In [ ]:
def spherical_kmeans(normalized_vectors, n_clusters, random_state, n_init=10, max_iter=100):
    rng = np.random.default_rng(random_state)
    best_labels, best_inertia = None, np.inf

    for _ in range(n_init):
        centers = normalized_vectors[rng.choice(len(normalized_vectors), n_clusters, replace=False)].copy()
        previous_labels = None
        for _ in range(max_iter):
            similarities = normalized_vectors @ centers.T
            labels = similarities.argmax(axis=1)
            if previous_labels is not None and np.array_equal(labels, previous_labels):
                break
            previous_labels = labels.copy()

            new_centers = np.zeros_like(centers)
            for cluster_id in range(n_clusters):
                members = normalized_vectors[labels == cluster_id]
                if len(members) == 0:
                    new_centers[cluster_id] = normalized_vectors[rng.integers(len(normalized_vectors))]
                else:
                    center = members.mean(axis=0)
                    new_centers[cluster_id] = center / max(np.linalg.norm(center), 1e-12)
            centers = new_centers

        similarities = normalized_vectors @ centers.T
        labels = similarities.argmax(axis=1)
        inertia = float(np.sum(1.0 - similarities[np.arange(len(labels)), labels]))
        if inertia < best_inertia:
            best_labels, best_inertia = labels.copy(), inertia

    return best_labels, best_inertia


def choose_elbow_k(normalized_vectors, max_k, seed):
    max_k = min(max_k, len(normalized_vectors))
    if max_k < 2:
        return 1, np.array([1]), np.array([0.0]), np.zeros(len(normalized_vectors), dtype=np.int64)
    ks = np.arange(2, max_k + 1)
    inertias, labels_by_k = [], {}
    for k in ks:
        labels, inertia = spherical_kmeans(normalized_vectors, int(k), random_state=seed)
        inertias.append(inertia)
        labels_by_k[int(k)] = labels
    points = np.column_stack([ks.astype(float), np.asarray(inertias)])
    normalized = (points - points.min(axis=0)) / np.maximum(points.max(axis=0) - points.min(axis=0), 1e-12)
    line = normalized[-1] - normalized[0]
    distances = np.abs(
        line[0] * (normalized[:, 1] - normalized[0, 1]) - line[1] * (normalized[:, 0] - normalized[0, 0])
    ) / max(np.linalg.norm(line), 1e-12)
    selected_k = int(ks[int(np.argmax(distances))])
    return selected_k, ks, np.asarray(inertias), labels_by_k[selected_k]


def compute_client_centroids(features_gpu, client_idx_list):
    centroids = []
    for idx in client_idx_list:
        vec = features_gpu[idx.to(features_gpu.device)].mean(dim=0)
        vec = vec / torch.linalg.vector_norm(vec).clamp_min(1e-12)
        centroids.append(vec.cpu().numpy())
    return np.stack(centroids, axis=0)


if EXPERIMENT_MODE == 'cluster_cskgo':
    client_centroids = compute_client_centroids(train_features, client_indices)
    selected_k_clients, elbow_ks, elbow_inertias, client_cluster_labels = choose_elbow_k(
        client_centroids, K_MAX, CLUSTER_SEED
    )
    # Remap cluster labels về 0..M-1 liên tục
    unique_labels = sorted(np.unique(client_cluster_labels).tolist())
    label_remap = {old: new for new, old in enumerate(unique_labels)}
    client_cluster_labels = np.array([label_remap[l] for l in client_cluster_labels], dtype=np.int64)
    NUM_CLUSTERS = len(unique_labels)
else:
    # global_cskgo: bỏ bước cluster, coi toàn bộ clients là 1 nhóm duy nhất
    client_cluster_labels = np.zeros(NUM_CLIENTS, dtype=np.int64)
    elbow_ks, elbow_inertias = np.array([1]), np.array([0.0])
    NUM_CLUSTERS = 1

cluster_member_clients = {
    m: np.flatnonzero(client_cluster_labels == m).tolist()
    for m in range(NUM_CLUSTERS)
}
cluster_indices = {
    m: torch.cat([client_indices[cid] for cid in cluster_member_clients[m]])
    for m in range(NUM_CLUSTERS)
}

cluster_class_counts_before = np.zeros((NUM_CLUSTERS, NUM_CLASSES), dtype=np.int64)
for m in range(NUM_CLUSTERS):
    y_m = train_labels[cluster_indices[m]]
    cluster_class_counts_before[m] = torch.bincount(y_m, minlength=NUM_CLASSES).numpy()

print(f'Client clustering ({EXPERIMENT_MODE}): NUM_CLUSTERS = {NUM_CLUSTERS}')
for m in range(NUM_CLUSTERS):
    print(f'  Cluster {m} (clients={cluster_member_clients[m]}, total={len(cluster_indices[m])}): '
          f'{cluster_class_counts_before[m].tolist()}')

if EXPERIMENT_MODE == 'cluster_cskgo':
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(elbow_ks, elbow_inertias, marker='o')
    axes[0].axvline(NUM_CLUSTERS, color='red', linestyle='--', label=f'Elbow K={NUM_CLUSTERS}')
    axes[0].set(xlabel='Number of client clusters (K)', ylabel='Cosine inertia', title='Client Elbow Curve')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    bottom = np.zeros(NUM_CLUSTERS)
    for c in range(NUM_CLASSES):
        vals = cluster_class_counts_before[:, c]
        axes[1].bar(np.arange(NUM_CLUSTERS), vals, bottom=bottom, label=f'C{c}')
        bottom += vals
    axes[1].axhline(CLUSTER_CLASS_THRESHOLD, color='black', linestyle='--', linewidth=1, label=f'T={CLUSTER_CLASS_THRESHOLD}')
    axes[1].set(xlabel='Cluster ID', ylabel='Sample count', title='Class Distribution per Cluster (Before CSKGO)')
    axes[1].legend(ncol=3, fontsize=8)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'client_clustering.png', dpi=150)
    plt.show()


In [ ]:
def _euclidean_kmeans_torch(X, n_clusters, seed, n_init=5, max_iter=50):
    """Chạy K-means Euclid trên tensor X (N, D) trả về labels, centers, empty_free."""
    n_samples = X.shape[0]
    best_labels, best_centers, best_sse = None, None, float('inf')

    for init_idx in range(n_init):
        gen = torch.Generator(device=X.device)
        gen.manual_seed(seed + init_idx * 9973)
        perm = torch.randperm(n_samples, generator=gen, device=X.device)
        centers = X[perm[:n_clusters]].clone()
        prev_labels = None

        for _ in range(max_iter):
            dists = torch.cdist(X, centers)
            labels = dists.argmin(dim=1)
            if prev_labels is not None and torch.equal(labels, prev_labels):
                break
            prev_labels = labels

            new_centers = torch.zeros_like(centers)
            for cid in range(n_clusters):
                mask = labels == cid
                if mask.any():
                    new_centers[cid] = X[mask].mean(dim=0)
                else:
                    # Gán lại tâm cụm rỗng về điểm xa tâm gần nhất nhất để tránh cụm suy biến
                    min_d = dists.min(dim=1).values
                    new_centers[cid] = X[int(min_d.argmax().item())]
            centers = new_centers

        final_dists = torch.cdist(X, centers)
        labels = final_dists.argmin(dim=1)
        sse = float((final_dists.min(dim=1).values ** 2).sum().item())
        if sse < best_sse:
            best_sse = sse
            best_labels = labels
            best_centers = centers

    return best_labels, best_centers


def _compute_csk_indices(X, labels, centers, beta=0.7):
    """Tính Com(C_i), comcs, sepcs và CVIN = comcs / sepcs theo CSKGO paper."""
    unique_ids = torch.unique(labels)
    k = len(unique_ids)
    if k < 2:
        return None

    # Remap cụm thực sự có mẫu để phòng trường hợp đặc biệt (ví dụ các điểm trùng hệt nhau)
    clusters = [X[labels == cid] for cid in unique_ids]
    active_centers = torch.stack([c.mean(dim=0) for c in clusters], dim=0)

    com_list = []
    for c_pts, center in zip(clusters, active_centers):
        d = torch.linalg.vector_norm(c_pts - center.unsqueeze(0), dim=1)
        com_list.append(float(d.mean().item()))

    com_sum = sum(com_list)
    if com_sum <= 1e-12:
        comcs = 1.0 / k
    else:
        comcs = max(com_list) / com_sum

    center_dists = torch.cdist(active_centers, active_centers)
    sep_matrix = torch.zeros((k, k), dtype=X.dtype, device=X.device)
    for j in range(k):
        # Khoảng cách mọi mẫu tới tâm cluster j
        dist_to_vj = torch.linalg.vector_norm(X - active_centers[j].unsqueeze(0), dim=1)
        for i in range(k):
            if i == j:
                continue
            d_ij = float(center_dists[i, j].item())
            radius = beta * d_ij
            mask_i = labels == unique_ids[i]
            # Eq. (3)-(4) có +1 ở mẫu số để tránh chia 0 khi 2 cụm tách rời hoàn toàn
            count_in_r = float((dist_to_vj[mask_i] <= radius).sum().item())
            sep_matrix[i, j] = d_ij / (1.0 + count_in_r)

    row_means = sep_matrix.sum(dim=1) / max(k - 1, 1)
    sepcs = float(row_means.min().item())
    cvin = comcs / max(sepcs, 1e-12)
    return {
        'k': k,
        'clusters': clusters,
        'com_list': com_list,
        'comcs': comcs,
        'sepcs': sepcs,
        'cvin': cvin,
    }


def csk_means(X_min, beta=0.7, seed=45, n_init=5, max_iter=50):
    """Tìm số cụm con tối ưu k* in [2, floor(sqrt(N))] theo cực tiểu CVIN."""
    n_samples = X_min.shape[0]
    k_max = int(np.floor(np.sqrt(n_samples)))
    best_res = None

    for k in range(2, k_max + 1):
        labels, centers = _euclidean_kmeans_torch(X_min, k, seed=seed + k * 101, n_init=n_init, max_iter=max_iter)
        res = _compute_csk_indices(X_min, labels, centers, beta=beta)
        if res is None:
            continue
        if best_res is None or res['cvin'] < best_res['cvin']:
            best_res = res

    if best_res is None:
        # Fallback an toàn nếu mọi điểm trùng nhau khiến không tách được >= 2 cụm
        d = torch.linalg.vector_norm(X_min - X_min.mean(dim=0, keepdim=True), dim=1)
        best_res = {
            'k': 1,
            'clusters': [X_min],
            'com_list': [float(d.mean().item())],
            'comcs': 1.0,
            'sepcs': 1.0,
            'cvin': 1.0,
        }
    return best_res


def allocate_sampling_ratios(clusters, com_list, n_to_generate):
    """Phân bổ số lượng mẫu cần sinh cho từng sub-cluster theo độ chặt Com(C_i)."""
    if n_to_generate <= 0:
        return [0] * len(clusters)

    pos_coms = [c for c, pts in zip(com_list, clusters) if len(pts) > 1 and c > 1e-8]
    max_com = max(pos_coms) if pos_coms else 1.0
    eps = max(0.1 * float(np.mean(pos_coms)), 1e-4) if pos_coms else 1e-4

    # Cụm chỉ có 1 điểm (singleton outlier) có Com=0 giả tạo -> gán mức phạt max_com để không hút hết mẫu sinh
    effective_coms = [
        max_com if len(pts) <= 1 else c
        for c, pts in zip(com_list, clusters)
    ]
    weights = np.array([1.0 / (c + eps) for c in effective_coms], dtype=np.float64)
    probs = weights / weights.sum()

    raw_counts = probs * n_to_generate
    int_counts = np.floor(raw_counts).astype(np.int64)
    remainder = int(n_to_generate - int_counts.sum())
    if remainder > 0:
        order = np.argsort(-(raw_counts - int_counts))
        for idx in order[:remainder]:
            int_counts[idx] += 1
    return int_counts.tolist()


def gaussian_oversample_cluster(samples, n_gen, cov_reg=1e-4, generator=None):
    """Sinh n_gen mẫu từ phân phối chuẩn đa biến N(mu, Sigma + cov_reg*I)."""
    if n_gen <= 0:
        return torch.empty((0, samples.shape[1]), dtype=samples.dtype, device=samples.device)

    n_samples, dim = samples.shape
    mu = samples.mean(dim=0, keepdim=True)
    z = torch.randn((n_gen, dim), generator=generator, dtype=samples.dtype, device=samples.device)

    if n_samples < 2:
        return mu + np.sqrt(cov_reg) * z

    centered = samples - mu
    cov = (centered.T @ centered) / float(n_samples)
    cov = 0.5 * (cov + cov.T)
    eye = torch.eye(dim, dtype=samples.dtype, device=samples.device)
    cov_reg_mat = cov + cov_reg * eye

    try:
        L = torch.linalg.cholesky(cov_reg_mat)
        return mu + z @ L.T
    except RuntimeError:
        eigvals, eigvecs = torch.linalg.eigh(cov_reg_mat)
        eigvals = eigvals.clamp_min(cov_reg)
        A = eigvecs * torch.sqrt(eigvals).unsqueeze(0)
        return mu + z @ A.T


@torch.no_grad()
def cskgo_oversample_class(
    X_class,
    target_count,
    min_samples_for_cskmeans=20,
    max_ratio=10,
    beta=0.7,
    cov_reg=1e-4,
    seed=45,
):
    """Chạy CSKGO cho 1 tập mẫu lớp thiểu số để đạt mục tiêu target_count."""
    n_orig = int(X_class.shape[0])
    if n_orig == 0:
        return torch.empty((0, X_class.shape[1]), dtype=X_class.dtype, device=X_class.device), {
            'n_orig': 0, 'n_target': 0, 'n_gen': 0, 'k_sub': 0, 'mode': 'empty'
        }

    effective_target = int(target_count) if max_ratio is None else min(int(target_count), int(max_ratio * n_orig))
    n_gen = max(0, effective_target - n_orig)
    if n_gen == 0:
        return torch.empty((0, X_class.shape[1]), dtype=X_class.dtype, device=X_class.device), {
            'n_orig': n_orig, 'n_target': n_orig, 'n_gen': 0, 'k_sub': 0, 'mode': 'sufficient'
        }

    gen = torch.Generator(device=X_class.device)
    gen.manual_seed(seed)

    if n_orig < min_samples_for_cskmeans:
        # < 20 mẫu: xem là 1 cụm (k=1) và áp dụng GOS bình thường
        syn = gaussian_oversample_cluster(X_class, n_gen, cov_reg=cov_reg, generator=gen)
        return syn, {
            'n_orig': n_orig, 'n_target': n_orig + n_gen, 'n_gen': n_gen,
            'k_sub': 1, 'mode': 'single_cluster_gos'
        }

    csk_res = csk_means(X_class, beta=beta, seed=seed)
    gen_counts = allocate_sampling_ratios(csk_res['clusters'], csk_res['com_list'], n_gen)
    syn_parts = [
        gaussian_oversample_cluster(sub_pts, count, cov_reg=cov_reg, generator=gen)
        for sub_pts, count in zip(csk_res['clusters'], gen_counts)
        if count > 0
    ]
    syn = torch.cat(syn_parts, dim=0) if syn_parts else torch.empty((0, X_class.shape[1]), dtype=X_class.dtype, device=X_class.device)
    return syn, {
        'n_orig': n_orig,
        'n_target': n_orig + int(syn.shape[0]),
        'n_gen': int(syn.shape[0]),
        'k_sub': int(csk_res['k']),
        'cvin': float(csk_res['cvin']),
        'mode': 'full_cskgo',
    }


In [ ]:
cskgo_start = time.time()
cluster_balanced_data = {}
cskgo_logs = []

if EXPERIMENT_MODE == 'global_cskgo':
    target_per_class = int(global_class_counts.max()) if GLOBAL_TARGET_COUNT is None else int(GLOBAL_TARGET_COUNT)
else:
    target_per_class = int(CLUSTER_CLASS_THRESHOLD)

for m in range(NUM_CLUSTERS):
    idx_m = cluster_indices[m]
    X_m_orig = train_features[idx_m.to(DEVICE)]
    y_m_orig = train_labels_gpu[idx_m.to(DEVICE)]

    X_parts = [X_m_orig]
    y_parts = [y_m_orig]

    for c in range(NUM_CLASSES):
        mask_c = y_m_orig == c
        X_mc = X_m_orig[mask_c]
        syn_mc, info = cskgo_oversample_class(
            X_mc,
            target_count=target_per_class,
            min_samples_for_cskmeans=MIN_SAMPLES_FOR_CSKMEANS,
            max_ratio=MAX_OVERSAMPLE_RATIO,
            beta=CSKGO_BETA,
            cov_reg=COV_REG,
            seed=CSKGO_SEED + m * 100 + c,
        )
        if syn_mc.shape[0] > 0:
            X_parts.append(syn_mc)
            y_parts.append(torch.full((syn_mc.shape[0],), c, dtype=torch.long, device=DEVICE))
        cskgo_logs.append({'cluster': m, 'class': c, **info})

    cluster_balanced_data[m] = (torch.cat(X_parts, dim=0), torch.cat(y_parts, dim=0))

cskgo_time_sec = time.time() - cskgo_start

cluster_class_counts_after = np.zeros((NUM_CLUSTERS, NUM_CLASSES), dtype=np.int64)
for m in range(NUM_CLUSTERS):
    _, y_m_bal = cluster_balanced_data[m]
    cluster_class_counts_after[m] = torch.bincount(y_m_bal.cpu(), minlength=NUM_CLASSES).numpy()

total_before = int(cluster_class_counts_before.sum())
total_after = int(cluster_class_counts_after.sum())
total_generated = total_after - total_before

print(f'CSKGO completed in {cskgo_time_sec:.3f}s | Before: {total_before:,} -> After: {total_after:,} (+{total_generated:,} synthetic)')
print('Class counts BEFORE (global sum):', cluster_class_counts_before.sum(axis=0).tolist())
print('Class counts AFTER  (global sum):', cluster_class_counts_after.sum(axis=0).tolist())

with CSKGO_STATS_PATH.open('w') as handle:
    json.dump({
        'experiment_mode': EXPERIMENT_MODE,
        'target_per_class': target_per_class,
        'cskgo_time_sec': cskgo_time_sec,
        'total_before': total_before,
        'total_after': total_after,
        'total_generated': total_generated,
        'per_cluster_class_logs': cskgo_logs,
    }, handle, indent=2)

# Biểu đồ so sánh phân bố class trước và sau CSKGO
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
classes_x = np.arange(NUM_CLASSES)
axes[0].bar(classes_x - 0.2, cluster_class_counts_before.sum(axis=0), width=0.4, label='Before CSKGO')
axes[0].bar(classes_x + 0.2, cluster_class_counts_after.sum(axis=0), width=0.4, label='After CSKGO')
axes[0].set(xticks=classes_x, xlabel='Class', ylabel='Total samples', title=f'Global Class Counts ({EXPERIMENT_MODE})')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

bottom = np.zeros(NUM_CLUSTERS)
for c in range(NUM_CLASSES):
    vals = cluster_class_counts_after[:, c]
    axes[1].bar(np.arange(NUM_CLUSTERS), vals, bottom=bottom, label=f'C{c}')
    bottom += vals
axes[1].set(xlabel='Cluster ID', ylabel='Sample count', title='Cluster Distributions (After CSKGO)')
axes[1].legend(ncol=3, fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RUN_DIR / 'class_dist_before_after.png', dpi=150)
plt.show()


In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(x)


def batch_seed(step_id, entity_id, local_epoch):
    return BATCH_SEED + step_id * 100003 + entity_id * 1009 + local_epoch * 97


def train_epoch_on_tensors(model, optimizer, X_data, y_data, seed):
    model.train()
    gen = torch.Generator(device=X_data.device)
    gen.manual_seed(seed)
    perm = torch.randperm(len(y_data), generator=gen, device=X_data.device)
    for start in range(0, len(perm), BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        loss = F.cross_entropy(model(X_data[idx]), y_data[idx])
        loss.backward()
        optimizer.step()


@torch.no_grad()
def evaluate_imbalanced(model, features, labels):
    model.eval()
    total_loss = 0.0
    conf_mat = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long, device=labels.device)
    for start in range(0, len(labels), EVAL_BATCH_SIZE):
        x = features[start:start + EVAL_BATCH_SIZE]
        y = labels[start:start + EVAL_BATCH_SIZE]
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction='sum').item()
        preds = logits.argmax(dim=1)
        flat_idx = y * NUM_CLASSES + preds
        conf_mat += torch.bincount(flat_idx, minlength=NUM_CLASSES * NUM_CLASSES).view(NUM_CLASSES, NUM_CLASSES)

    cm = conf_mat.cpu().numpy().astype(np.float64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)

    recalls = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precisions = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1s = np.divide(2 * precisions * recalls, precisions + recalls, out=np.zeros_like(tp), where=(precisions + recalls) > 0)

    overall_acc = float(tp.sum() / max(cm.sum(), 1.0))
    balanced_acc = float(recalls.mean())
    macro_f1 = float(f1s.mean())
    g_mean = float(np.exp(np.mean(np.log(np.clip(recalls, 1e-12, 1.0)))))
    minority_rec = float(recalls[MINORITY_CLASSES].mean()) if MINORITY_CLASSES else balanced_acc
    majority_rec = float(recalls[MAJORITY_CLASSES].mean()) if MAJORITY_CLASSES else balanced_acc

    return {
        'loss': total_loss / len(labels),
        'accuracy': overall_acc,
        'balanced_accuracy': balanced_acc,
        'macro_f1': macro_f1,
        'g_mean': g_mean,
        'minority_recall': minority_rec,
        'majority_recall': majority_rec,
        'per_class_recall': recalls.tolist(),
    }


def fedavg_states(states_list, sizes_list):
    total_size = float(sum(sizes_list))
    agg = {}
    for name in states_list[0]:
        agg[name] = sum(state[name] * (size / total_size) for state, size in zip(states_list, sizes_list))
    return agg


seed_everything(MODEL_SEED)
global_model = LinearProbe(EMBEDDING_DIM, NUM_CLASSES).to(DEVICE)

X_server_all = torch.cat([cluster_balanced_data[m][0] for m in range(NUM_CLUSTERS)], dim=0)
y_server_all = torch.cat([cluster_balanced_data[m][1] for m in range(NUM_CLUSTERS)], dim=0)

METRIC_FIELDS = [
    'step', 'train_loss', 'train_accuracy', 'test_loss', 'test_accuracy',
    'balanced_accuracy', 'macro_f1', 'g_mean', 'minority_recall', 'majority_recall',
]
history_rows = []

if TRAIN_PARADIGM == 'centralized':
    optimizer = torch.optim.SGD(global_model.parameters(), lr=LEARNING_RATE)
    total_steps = EPOCHS
    for epoch in range(1, total_steps + 1):
        t0 = time.time()
        train_epoch_on_tensors(global_model, optimizer, X_server_all, y_server_all, batch_seed(epoch, 0, epoch))
        train_res = evaluate_imbalanced(global_model, X_server_all, y_server_all)
        test_res = evaluate_imbalanced(global_model, test_features, test_labels_gpu)
        row = {
            'step': epoch,
            'train_loss': train_res['loss'],
            'train_accuracy': train_res['accuracy'],
            'test_loss': test_res['loss'],
            'test_accuracy': test_res['accuracy'],
            'balanced_accuracy': test_res['balanced_accuracy'],
            'macro_f1': test_res['macro_f1'],
            'g_mean': test_res['g_mean'],
            'minority_recall': test_res['minority_recall'],
            'majority_recall': test_res['majority_recall'],
        }
        history_rows.append(row)
        print(
            f"Epoch {epoch:02d}/{total_steps} | test_acc={row['test_accuracy']:.4f} | "
            f"bal_acc={row['balanced_accuracy']:.4f} | macro_f1={row['macro_f1']:.4f} | "
            f"min_rec={row['minority_recall']:.4f} | {time.time()-t0:.2f}s"
        )
else:
    worker_model = LinearProbe(EMBEDDING_DIM, NUM_CLASSES).to(DEVICE)
    total_steps = ROUNDS
    for round_id in range(1, total_steps + 1):
        t0 = time.time()
        global_state = {k: v.detach().clone() for k, v in global_model.named_parameters()}
        cluster_states, cluster_sizes = [], []

        for m in range(NUM_CLUSTERS):
            X_m, y_m = cluster_balanced_data[m]
            worker_model.load_state_dict(global_state, strict=True)
            optimizer = torch.optim.SGD(worker_model.parameters(), lr=LEARNING_RATE)
            for le in range(1, LOCAL_EPOCHS + 1):
                train_epoch_on_tensors(worker_model, optimizer, X_m, y_m, batch_seed(round_id, m, le))
            cluster_states.append({k: v.detach().clone() for k, v in worker_model.named_parameters()})
            cluster_sizes.append(len(y_m))

        new_state = fedavg_states(cluster_states, cluster_sizes)
        global_model.load_state_dict(new_state, strict=True)

        train_res = evaluate_imbalanced(global_model, X_server_all, y_server_all)
        test_res = evaluate_imbalanced(global_model, test_features, test_labels_gpu)
        row = {
            'step': round_id,
            'train_loss': train_res['loss'],
            'train_accuracy': train_res['accuracy'],
            'test_loss': test_res['loss'],
            'test_accuracy': test_res['accuracy'],
            'balanced_accuracy': test_res['balanced_accuracy'],
            'macro_f1': test_res['macro_f1'],
            'g_mean': test_res['g_mean'],
            'minority_recall': test_res['minority_recall'],
            'majority_recall': test_res['majority_recall'],
        }
        history_rows.append(row)
        print(
            f"Round {round_id:02d}/{total_steps} | test_acc={row['test_accuracy']:.4f} | "
            f"bal_acc={row['balanced_accuracy']:.4f} | macro_f1={row['macro_f1']:.4f} | "
            f"min_rec={row['minority_recall']:.4f} | {time.time()-t0:.2f}s"
        )

with METRICS_PATH.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
    writer.writeheader()
    writer.writerows(history_rows)

torch.save({k: v.detach().cpu() for k, v in global_model.state_dict().items()}, FINAL_MODEL_PATH)
print('Saved metrics to:', METRICS_PATH)


In [ ]:
final_eval = evaluate_imbalanced(global_model, test_features, test_labels_gpu)

config_payload = {
    'run_id': RUN_ID,
    'experiment_mode': EXPERIMENT_MODE,
    'train_paradigm': TRAIN_PARADIGM,
    'dataset_name': DATASET_NAME,
    'imbalance_mode': IMBALANCE_MODE,
    'partition_type': PARTITION_TYPE,
    'dirichlet_alpha': DIRICHLET_ALPHA,
    'num_clients': NUM_CLIENTS,
    'num_clusters': NUM_CLUSTERS,
    'resnet_size': RESNET_SIZE,
    'k_max': K_MAX,
    'cluster_class_threshold': CLUSTER_CLASS_THRESHOLD,
    'global_target_count': GLOBAL_TARGET_COUNT,
    'min_samples_for_cskmeans': MIN_SAMPLES_FOR_CSKMEANS,
    'max_oversample_ratio': MAX_OVERSAMPLE_RATIO,
    'cskgo_beta': CSKGO_BETA,
    'cov_reg': COV_REG,
    'epochs': EPOCHS,
    'rounds': ROUNDS,
    'local_epochs': LOCAL_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'partition_seed': PARTITION_SEED,
    'model_seed': MODEL_SEED,
    'batch_seed': BATCH_SEED,
    'cluster_seed': CLUSTER_SEED,
    'cskgo_seed': CSKGO_SEED,
}
with CONFIG_PATH.open('w') as handle:
    json.dump(config_payload, handle, indent=2)

summary_payload = {
    'run_id': RUN_ID,
    'experiment_mode': EXPERIMENT_MODE,
    'train_paradigm': TRAIN_PARADIGM,
    'num_clusters': NUM_CLUSTERS,
    'cskgo_time_sec': cskgo_time_sec,
    'total_before': total_before,
    'total_after': total_after,
    'total_generated': total_generated,
    'final_test_accuracy': final_eval['accuracy'],
    'final_balanced_accuracy': final_eval['balanced_accuracy'],
    'final_macro_f1': final_eval['macro_f1'],
    'final_g_mean': final_eval['g_mean'],
    'final_minority_recall': final_eval['minority_recall'],
    'final_majority_recall': final_eval['majority_recall'],
    'per_class_recall': final_eval['per_class_recall'],
}
with SUMMARY_PATH.open('w') as handle:
    json.dump(summary_payload, handle, indent=2)

steps = [r['step'] for r in history_rows]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(steps, [r['test_accuracy'] for r in history_rows], label='Test Acc')
axes[0].plot(steps, [r['macro_f1'] for r in history_rows], label='Macro-F1')
axes[0].plot(steps, [r['minority_recall'] for r in history_rows], label='Minority Recall')
axes[0].set(xlabel='Step (Epoch/Round)', ylabel='Score', title=f'{EXPERIMENT_MODE} ({TRAIN_PARADIGM})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, [r['train_loss'] for r in history_rows], label='Train Loss')
axes[1].plot(steps, [r['test_loss'] for r in history_rows], label='Test Loss')
axes[1].set(xlabel='Step (Epoch/Round)', ylabel='Loss', title='Losses')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

classes_x = np.arange(NUM_CLASSES)
bar_colors = ['tab:red' if c in MINORITY_CLASSES else 'tab:blue' for c in range(NUM_CLASSES)]
axes[2].bar(classes_x, final_eval['per_class_recall'], color=bar_colors)
axes[2].set(xticks=classes_x, ylim=(0, 1.0), xlabel='Class (Red = Minority)', ylabel='Recall', title='Per-Class Test Recall')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RUN_DIR / 'metrics.png', dpi=150)
plt.show()

print('Summary saved to:', SUMMARY_PATH)
print(json.dumps(summary_payload, indent=2))

# ── So sánh nhiều RUN_ID trong cskgo_stage1/ ──────────────────────────────────
COMPARE_RUN_IDS = [RUN_ID]  # Ví dụ: [1, 2, 3]
print('\n=== Run Comparison ===')
for rid in COMPARE_RUN_IDS:
    rdir = EXP_DIR / (str(rid) if str(rid).startswith('run') else f'run{rid}')
    spath = rdir / 'summary.json'
    if not spath.exists():
        print(f'  Run {rid}: missing summary.json')
        continue
    with spath.open() as handle:
        s = json.load(handle)
    print(
        f"  Run {rid} [{s['experiment_mode']}/{s['train_paradigm']}] | K={s['num_clusters']} | "
        f"CSKGO={s['cskgo_time_sec']:.2f}s | Acc={s['final_test_accuracy']:.4f} | "
        f"BalAcc={s['final_balanced_accuracy']:.4f} | MacroF1={s['final_macro_f1']:.4f} | "
        f"MinRec={s['final_minority_recall']:.4f}"
    )
